# Incident reports — generate a run & explore

This notebook can **trigger a full ingestion run** and then **explore** its
output. It **reuses** the logic from `src/` and never duplicates it.

- **Section A** reproduces the standard production graphs by calling `src/`.
- **Section B** adds extra ad-hoc analyses, inline.

## 0. Setup

In [ ]:
import sys
from pathlib import Path

%load_ext autoreload
%autoreload 2

cwd = Path.cwd()
PROJECT_ROOT = cwd.parent if cwd.name == "notebooks" else cwd
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import Image, display

%matplotlib inline

from src import config
print("Project root:", PROJECT_ROOT)

## 1. Generate a fresh run (or pick the latest)

`run_from_env` runs the whole pipeline and writes all artifacts into a new
timestamped folder (same as the CLI).

- `GENERATE_NEW_RUN = True` -> create a brand-new run.
- `GENERATE_NEW_RUN = False` -> only explore the most recent existing run.

In [ ]:
from src.sources.incidents.runner import run_from_env

GENERATE_NEW_RUN = False  # True = create a new run; False = explore the latest

if GENERATE_NEW_RUN:
    run_dir = run_from_env(config.DEFAULT_INPUT_CSV)
else:
    runs = sorted(
        p for p in config.ARTIFACTS_DIR.iterdir() if p.is_dir() and p.name.isdigit()
    )
    assert runs, "No run found. Set GENERATE_NEW_RUN = True to create one."
    run_dir = runs[-1]

RUN_ID = run_dir.name
print("Active run:", RUN_ID)

## 2. Load the anonymised dataset

In [ ]:
df = pd.read_csv(run_dir / "incidents_anonymized.csv", parse_dates=[config.DATE_COLUMN])
df.head()

## 3. Dataset overview

In [ ]:
print("Shape:", df.shape)
df.info()

In [ ]:
df.describe(include="all").T

## 4. Quality metrics (reuse `src/`)

Same function as the pipeline, so the numbers match the registry.

In [ ]:
from src.common.metrics import compute_quality_metrics

compute_quality_metrics(df)

## A. Standard production graphs (via `src/`)

These cells call the **exact functions** used at each run and display the saved
PNGs inline. They regenerate the files inside `run_dir`.

> To avoid overwriting a committed run's PNGs, point the functions to a scratch
> folder instead of `run_dir`.

### A.1 Temporal distributions — `1.x`

Incidents per day / week / shift.

In [ ]:
from src.sources.incidents import distributions

for png_path in distributions.plot_all(df, run_dir):
    display(Image(filename=str(png_path)))

### A.2 Incident histograms — `2.x`

Incidents per machine / operator / signal / confidence index.

In [ ]:
from src.sources.incidents import histograms

for png_path in histograms.plot_all(df, run_dir):
    display(Image(filename=str(png_path)))

### A.3 Correlations — `3.x`

Severity / signals and severity / comment presence.

In [ ]:
from src.sources.incidents import correlations

for png_path in correlations.plot_all(df, run_dir):
    display(Image(filename=str(png_path)))

## B. Additional inline analyses

Ad-hoc explorations not part of the standard production set. Computed and
plotted inline (no PNG written).

### B.1 Incidents by severity and machine (sorted by severity 5 -> 1)

In [ ]:
ct_machine = pd.crosstab(df[config.MACHINE_COLUMN], df[config.SEVERITY_COLUMN])
severity_order = sorted(ct_machine.columns, reverse=True)
ct_machine = ct_machine.sort_values(by=severity_order, ascending=False)
display(ct_machine)

ax = ct_machine.plot.bar(stacked=True, figsize=(12, 5), colormap="viridis")
ax.set_xlabel("Machine")
ax.set_ylabel("Number of incidents")
ax.set_title("Incidents by machine and severity (sorted by severity 5 -> 1)")
ax.legend(title="severity", bbox_to_anchor=(1.02, 1), loc="upper left")
plt.tight_layout()
plt.show()

### B.2 Incidents by severity and operator (sorted, top 10)

In [ ]:
ct_operator = pd.crosstab(df["operator_name"], df[config.SEVERITY_COLUMN])
severity_order = sorted(ct_operator.columns, reverse=True)
top_operators = ct_operator.sort_values(by=severity_order, ascending=False).head(10)
display(top_operators)

ax = top_operators.plot.bar(stacked=True, figsize=(12, 5), colormap="viridis")
ax.set_xlabel("Operator (pseudonymised)")
ax.set_ylabel("Number of incidents")
ax.set_title("Incidents by operator and severity (sorted by severity 5 -> 1, top 10)")
ax.legend(title="severity", bbox_to_anchor=(1.02, 1), loc="upper left")
plt.tight_layout()
plt.show()

### B.3 Incidents by shift and severity

In [ ]:
ct_shift = pd.crosstab(df[config.SHIFT_COLUMN], df[config.SEVERITY_COLUMN])
display(ct_shift)

ax = ct_shift.plot.bar(figsize=(10, 5), colormap="viridis")
ax.set_xlabel("Shift")
ax.set_ylabel("Number of incidents")
ax.set_title("Incidents by shift and severity")
ax.legend(title="severity")
plt.tight_layout()
plt.show()

### B.4 Which operators use which machines

In [ ]:
ct_op_machine = pd.crosstab(df["operator_name"], df[config.MACHINE_COLUMN])
display(ct_op_machine)

fig, ax = plt.subplots(figsize=(12, 6))
im = ax.imshow(ct_op_machine.values, aspect="auto", cmap="YlOrRd")
ax.set_xticks(range(ct_op_machine.shape[1]))
ax.set_xticklabels(ct_op_machine.columns, rotation=90)
ax.set_yticks(range(ct_op_machine.shape[0]))
ax.set_yticklabels([op[:8] for op in ct_op_machine.index])
ax.set_xlabel("Machine")
ax.set_ylabel("Operator (pseudonymised)")
ax.set_title("Operator x machine usage (incident counts)")
for i in range(ct_op_machine.shape[0]):
    for j in range(ct_op_machine.shape[1]):
        ax.text(j, i, ct_op_machine.values[i, j], ha="center", va="center", fontsize=7)
fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04, label="Incident count")
plt.tight_layout()
plt.show()

### B.5 Average reporting confidence per machine

In [ ]:
by_machine = (
    df.groupby(config.MACHINE_COLUMN)[config.CONFIDENCE_COLUMN].mean().sort_values()
)
ax = by_machine.plot.barh(figsize=(8, 5), color="#4C72B0")
ax.set_xlabel("Mean confidence index")
ax.set_title("Average reporting confidence per machine")
plt.tight_layout()
plt.show()

### B.6 Are severity and comment correlated?

Treats the comment as a **category** and tests whether a given comment maps
to a consistent severity (chi-square test + Cramer's V). The heatmap of this
association is produced in section A.3 (`3.2`).

In [ ]:
from src.sources.incidents.correlations import severity_comment_association

res = severity_comment_association(df)
print(f"Cramer's V  = {res['cramers_v']:.2f}")
print(f"chi-square  = {res['chi2']:.1f}  (dof = {res['dof']})")
print(f"p-value     = {res['p_value']:.2e}")
print(f"Verdict     : severity and comment are {res['verdict']}")

print("\nContingency table (comment x severity):")
display(res["table"])

### B.7 Are signals and severity correlated?

Same association analysis (chi-square test + Cramer's V) applied to **each**
`type_` signal vs severity. A high Cramer's V means that signal's presence is
strongly associated with the severity level.

In [ ]:
from scipy.stats import chi2_contingency

def _cramers_v_verdict(table):
    chi2, p, dof, _ = chi2_contingency(table.values)
    n = table.values.sum()
    r, k = table.shape
    denom = n * min(r - 1, k - 1)
    v = (chi2 / denom) ** 0.5 if denom > 0 else float('nan')
    strength = (
        'strong' if v >= 0.4 else 'moderate' if v >= 0.2 else 'weak' if v >= 0.1 else 'negligible'
    )
    sig = 'significant' if p < 0.05 else 'not significant'
    return v, chi2, dof, p, f'{strength}, {sig}'

present_signals = [c for c in config.SIGNAL_COLUMNS if c in df.columns]
rows = []
for s in present_signals:
    table = pd.crosstab(df[s], df[config.SEVERITY_COLUMN])
    v, chi2, dof, p, verdict = _cramers_v_verdict(table)
    rows.append(
        {
            'signal': s.replace('type_', ''),
            'cramers_v': round(v, 3),
            'chi2': round(chi2, 1),
            'dof': dof,
            'p_value': p,
            'verdict': verdict,
        }
    )

summary = pd.DataFrame(rows).sort_values('cramers_v', ascending=False).reset_index(drop=True)
display(summary)

ax = summary.set_index('signal')['cramers_v'].plot.barh(figsize=(8, 5), color='#C44E52')
ax.invert_yaxis()
ax.set_xlabel("Cramer's V (association with severity)")
ax.set_title('Signal vs severity association strength')
plt.tight_layout()
plt.show()

## C. Promoting an analysis to production

When a Section B cell proves useful and stable, move its logic into `src/`
(e.g. a function in `src/visualization/`), wire it into
`src/ingestion/runner.py`, and it becomes a standard artifact at every run.